# Machine Name Check

Check that the machines in the Excel file are in the full list of machines. If there are machines in the Excel file that are not in the full list of machines, this is an error and should be investigated. This could be due to a typo in the machine name or a machine that is not being tracked properly.

- **[Previous](https://app.fabric.microsoft.com/links/smyOMozGdW?ctid=ceadc058-fad7-4b6b-830b-00ac739654f0&pbi_source=linkShare&experience=fabric-developer&subfolderId=179239)**

In [5]:
import pandas as pd
import fsspec 
import yaml
# https://www.reddit.com/r/MicrosoftFabric/comments/1jwydwf/opening_file_with_python_without_need_for_mounted/
# Define lakehouse path and file location
lakehouse_abfss = "abfss://fad2d397-2419-4518-aa81-b0d383e4f42d@onelake.dfs.fabric.microsoft.com/49c6ae5f-f94f-47bb-9cbf-c0ee92ed80c1"
file_location = "Files/OAMForMonth.xlsx"

# Combine to create full path
excel_path = f"{lakehouse_abfss}/{file_location}"

# Read Excel file with pandas
df = pd.read_excel(excel_path,sheet_name=0)

# Rename all 35 columns the sheet date. get_sheet_date will not work after this because it relies on the columns data to work.
df.columns = ['c1','c2','c3','c4','c5','c6','c7','c8','c9','c10',
            'c11','c12','c13','c14','c15','c16','c17','c18','c19','c20',
            'c21','c22','c23','c24','c25','c26','c27','c28','c29','c30',
            'c31','c32','c33','c34','c35']
# print(df.head(5))

df_dropped_header_row = df.drop(0)
# verify header row is dropped and column headings are good.
# print(df_dropped_header_row.head(5))
# Drop rows with null values in column 'c1'
new_df = df_dropped_header_row.dropna(subset=['c1'])
# print(new_df.head(15))

# Define lakehouse path and file location
machine_list = "Files/machine_list.csv"
# Combine to create full path
machine_list_path = f"{lakehouse_abfss}/{machine_list}"
print(machine_list_path)

#deinf filesystem
filesystem_code = "abfss"
storage_options = {
        "account_name": "onelake",
        "account_host": "onelake.dfs.fabric.microsoft.com"
    }
onelake_fs = fsspec.filesystem(filesystem_code, **storage_options)

with onelake_fs.open(machine_list_path, "r") as file:
    # config = yaml.safe_load(file)
    a = file.read().splitlines()

# print(a)

# Check that the machines in the Excel file are in the full list of machines. If there are machines in the Excel file that are not in the full list of machines, this is an error and should be investigated. This could be due to a typo in the machine name or a machine that is not being tracked properly.
# with open(machine_list_path, 'r') as file:
#     a = file.read().splitlines()
# print(a)
# Using apply + lambda for 'name'
invalid_machines = new_df[new_df['c1'].apply(lambda x: x not in a)]
print("Rows where the machine name is not in the full list of names:")
# Notify owner that the following machines are not in the full list of machines. This is an error and should be investigated.
print(invalid_machines)

# Define lakehouse path and file location
machine_list_subset = "Files/machine_list_subset.csv"
# Combine to create full path
machine_list_subset_path = f"{lakehouse_abfss}/{machine_list_subset}"
print(machine_list_subset_path)

# # This time we will check the Excel file against the subset list of machines. This is not an error check but is a way to check that the code can identify machines that are not in the subset list. 
with onelake_fs.open(machine_list_subset_path, "r") as file:
    b = file.read().splitlines()

# with open('machine_list_subset.csv', 'r') as file:
#     b = file.read().splitlines()

not_errors = new_df[new_df['c1'].apply(lambda x: x not in b)]
print("Rows where the machine name is not in the subset list:")
print(not_errors['c1'])



abfss://fad2d397-2419-4518-aa81-b0d383e4f42d@onelake.dfs.fabric.microsoft.com/49c6ae5f-f94f-47bb-9cbf-c0ee92ed80c1/Files/machine_list.csv
Rows where the machine name is not in the full list of names:
Empty DataFrame
Columns: [c1, c2, c3, c4, c5, c6, c7, c8, c9, c10, c11, c12, c13, c14, c15, c16, c17, c18, c19, c20, c21, c22, c23, c24, c25, c26, c27, c28, c29, c30, c31, c32, c33, c34, c35]
Index: []

[0 rows x 35 columns]
abfss://fad2d397-2419-4518-aa81-b0d383e4f42d@onelake.dfs.fabric.microsoft.com/49c6ae5f-f94f-47bb-9cbf-c0ee92ed80c1/Files/machine_list_subset.csv
Rows where the machine name is not in the subset list:
2           SW-M2-HYD
9          SW-M2- Way
16         SW-M2-SPIN
23         OKK-M2-HYD
36          SW-M2-HYD
40          SW-M2-Way
44         SW-M2-SPIN
48      Nigata-M2-Hyd
57      Nigata-M2-Way
66     Nigata-M2-Spin
88         OKK-M2-Hyd
90         OKK-M2-Hyd
101      Mazak-M2-Hyd
108      Mazak-M2-Way
115     Mazak-M2-Spin
Name: c1, dtype: object
